In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install timm albumentations

In [ ]:
import os

# 1. Nạp Token và Tải Data
os.environ['KAGGLE_API_TOKEN'] = "KGAT_eee51873186b5d8faffb87b5bbf56416"

print("Đang tải dữ liệu từ Kaggle. Có thể mất vài phút...")
# Lệnh tải bộ dữ liệu BRISC từ Kaggle
!kaggle datasets download -d briscdataset/brisc2025

# Lệnh giải nén dữ liệu vào thư mục brisc_data
!unzip -q brisc2025.zip -d brisc_data
print("✅ Hoàn tất tải và giải nén!")

os.environ['BRISC_DATA_ROOT'] = '/content/brisc_data/brisc2025'

# 2. Kiểm tra cấu trúc
dest = '/content/brisc_data/brisc2025'
expected = [
    os.path.join(dest, "classification_task", "train"),
    os.path.join(dest, "segmentation_task",   "train", "images"),
    os.path.join(dest, "segmentation_task",   "train", "masks"),
]

print("\n── Kiểm tra đường dẫn ──")
ok = True
for path in expected:
    exists = os.path.isdir(path)
    symbol = "✓" if exists else "✗"
    print(f"  {symbol} {path}")
    if not exists: ok = False

if ok:
    print("\n✓ Cấu trúc chuẩn xác! Có thể chuyển qua bước Train.")
else:
    print("\n⚠ Cảnh báo: Sai cấu trúc! Hãy thử kiểm tra lệnh !ls /content/brisc_data")




Đang tải dữ liệu từ Kaggle. Có thể mất vài phút...
Dataset URL: https://www.kaggle.com/datasets/briscdataset/brisc2025
License(s): Attribution 4.0 International (CC BY 4.0)
100% 250M/250M [00:01<00:00, 132MB/s]

✅ Hoàn tất tải và giải nén!

── Kiểm tra đường dẫn ──
  ✓ /content/brisc_data/brisc2025/classification_task/train
  ✓ /content/brisc_data/brisc2025/segmentation_task/train/images
  ✓ /content/brisc_data/brisc2025/segmentation_task/train/masks

✓ Cấu trúc chuẩn xác! Có thể chuyển qua bước Train.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from copy import deepcopy
from tqdm import tqdm
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, random_split

import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.model_selection import train_test_split

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [ ]:
# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
DATA_ROOT = os.environ.get("BRISC_DATA_ROOT", "/content/brisc_data/brisc2025")

CLS_TRAIN_DIR = os.path.join(DATA_ROOT, "classification_task", "train")
CLS_TEST_DIR  = os.path.join(DATA_ROOT, "classification_task", "test")

OUTPUT_DIR = "/content/drive/MyDrive/brisc_project/outputs/classification"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAMES = ["glioma", "meningioma", "no_tumor", "pituitary"]
CLS_BATCH_SIZE   = 32
LEARNING_RATE    = 1e-4
WEIGHT_DECAY     = 1e-5
NUM_WORKERS      = 2

In [ ]:
# ==============================================================================
# 2. METRICS (ACCURACY, F1-SCORE & AGGREGATION)
# ==============================================================================
def compute_cls_metrics(all_preds: list, all_labels: list, class_names: list = CLASS_NAMES) -> dict:
    """
    Trả về dict gồm precision/recall/f1/accuracy (per-class + macro + weighted)
    """
    results = {}
    p = precision_score(all_labels, all_preds, average=None, labels=list(range(len(class_names))), zero_division=0)
    r = recall_score(all_labels, all_preds, average=None, labels=list(range(len(class_names))), zero_division=0)
    f = f1_score(all_labels, all_preds, average=None, labels=list(range(len(class_names))), zero_division=0)

    for i, cls in enumerate(class_names):
        results[cls] = {"precision": p[i], "recall": r[i], "f1": f[i]}

    results["macro"] = {
        "precision": precision_score(all_labels, all_preds, average="macro", zero_division=0),
        "recall":    recall_score(all_labels, all_preds, average="macro", zero_division=0),
        "f1":        f1_score(all_labels, all_preds, average="macro", zero_division=0),
    }

    results["weighted"] = {
        "precision": precision_score(all_labels, all_preds, average="weighted", zero_division=0),
        "recall":    recall_score(all_labels, all_preds, average="weighted", zero_division=0),
        "f1":        f1_score(all_labels, all_preds, average="weighted", zero_division=0),
        "accuracy":  accuracy_score(all_labels, all_preds),
    }
    return results
'''
def aggregate_runs(run_results: list) -> dict:
    """
    Nhận list kết quả qua 3 runs, tính Mean ± Std cho từng metric.
    """
    keys_top   = [k for k in run_results[0].keys() if isinstance(run_results[0][k], dict)]
    aggregated = {}

    for k in keys_top:
        sub_keys = list(run_results[0][k].keys())
        aggregated[k] = {}
        for sk in sub_keys:
            vals = [r[k][sk] for r in run_results]
            aggregated[k][sk] = {
                "mean": float(np.mean(vals)),
                "std":  float(np.std(vals)),
            }
    return aggregated

def print_cls_table(model_name: str, agg: dict):
    """In bảng kết quả giống Table 4 của paper (Mean ± Std)."""
    print(f"\n{'='*75}")
    print(f"  Model: {model_name} (Kết quả tổng hợp sau 3 lần chạy)")
    print(f"{'='*75}")
    header = f"  {'Class':<15} {'Precision':>14} {'Recall':>14} {'F1-Score':>14} {'Accuracy':>14}"
    print(header)
    print(f"  {'-'*71}")

    for cls in CLASS_NAMES + ["macro", "weighted"]:
        m = agg.get(cls, {})
        p  = f"{m['precision']['mean']*100:.2f}±{m['precision']['std']*100:.2f}" if 'precision' in m else "-"
        r  = f"{m['recall']['mean']*100:.2f}±{m['recall']['std']*100:.2f}"       if 'recall'    in m else "-"
        f1 = f"{m['f1']['mean']*100:.2f}±{m['f1']['std']*100:.2f}"               if 'f1'        in m else "-"
        ac = f"{m['accuracy']['mean']*100:.2f}±{m['accuracy']['std']*100:.2f}"   if 'accuracy'  in m else "—"

        print(f"  {cls:<15} {p:>14} {r:>14} {f1:>14} {ac:>14}")
    print()
'''

def print_cls_table(model_name: str, results: dict):
    """In bảng kết quả 1 lần chạy."""
    print(f"\n{'='*75}")
    print(f"  KẾT QUẢ TEST: {model_name.upper()}")
    print(f"{'='*75}")
    header = f"  {'Class':<15} {'Precision':>14} {'Recall':>14} {'F1-Score':>14} {'Accuracy':>14}"
    print(header)
    print(f"  {'-'*71}")

    for cls in CLASS_NAMES + ["macro", "weighted"]:
        m = results.get(cls, {})
        p  = f"{m.get('precision', 0.0)*100:.2f}%" if 'precision' in m else "-"
        r  = f"{m.get('recall', 0.0)*100:.2f}%"    if 'recall'    in m else "-"
        f1 = f"{m.get('f1', 0.0)*100:.2f}%"        if 'f1'        in m else "-"
        ac = f"{m.get('accuracy', 0.0)*100:.2f}%"  if 'accuracy'  in m else "—"

        print(f"  {cls:<15} {p:>14} {r:>14} {f1:>14} {ac:>14}")
    print(f"{'='*75}\n")

In [ ]:
# ==============================================================================
# 3. MODELS (TIMM)
# ==============================================================================

TIMM_NAME_MAP = {
    "resnet50":             "resnet50",
    "resnet101":            "resnet101",
    "densenet121":          "densenet121",
    "densenet169":          "densenet169",
    "mobilenetv2_100":      "mobilenetv2_100",
    "mobilenetv3_large_100":"mobilenetv3_large_100",
    "efficientnet_b0":      "efficientnet_b0",
    "efficientnet_b1":      "efficientnet_b1",
    "efficientnet_b2":      "efficientnet_b2",
    "xception":             "xception",
    "vgg16":                "vgg16",
    "vgg19":                "vgg19",
    "inception_v3":         "inception_v3",
}

# Inception V3 cần input size 299×299
INCEPTION_SIZE = 299

def build_cls_model(model_key: str, num_classes: int = 4,
                    pretrained: bool = True) -> nn.Module:
    """
    Khởi tạo model classification với head thay thế num_classes.

    Args:
        model_key   : một trong TIMM_NAME_MAP keys
        num_classes : số class (4 với BRISC)
        pretrained  : load ImageNet pretrained weights

    Returns:
        nn.Module sẵn sàng fine-tune
    """
    timm_name = TIMM_NAME_MAP.get(model_key, model_key)
    model = timm.create_model(timm_name,
                              pretrained=pretrained,
                              num_classes=num_classes)
    return model


def get_input_size(model_key: str) -> int:
    """Trả về image size phù hợp (Inception cần 299, còn lại 224)."""
    if model_key == "inception_v3":
        return INCEPTION_SIZE
    return 224

In [ ]:
# ==============================================================================
# 4. DATASET & AUGMENTATION
# ==============================================================================
def get_cls_transforms(image_size: int, split: str):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    if split == "train":
        return A.Compose([
            A.Resize(image_size, image_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.2),
            A.Rotate(limit=15, p=0.4),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
            A.GaussNoise(p=0.2),
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(image_size, image_size),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ])

class BRISCClassificationDataset(Dataset):
    """
    Cấu trúc thư mục:
        <root>/
          glioma/       *.jpg
          meningioma/   *.jpg
          no_tumor/     *.jpg
          pituitary/    *.jpg
    """
    CLASS_NAMES = ["glioma", "meningioma", "no_tumor", "pituitary"]
    CLASS2IDX   = {c: i for i, c in enumerate(CLASS_NAMES)}

    def __init__(self, root: str, split: str = "train", image_size: int = 224):
        self.root       = root
        self.split      = split
        self.image_size = image_size
        self.transform  = get_cls_transforms(image_size, split)

        self.samples: list[tuple[str, int]] = []
        for cls in self.CLASS_NAMES:
            cls_dir = os.path.join(root, cls)
            if not os.path.isdir(cls_dir):
                continue
            for fname in sorted(os.listdir(cls_dir)):
                if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    self.samples.append((os.path.join(cls_dir, fname), self.CLASS2IDX[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = cv2.imread(path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        augmented = self.transform(image=image)
        return augmented["image"], torch.tensor(label, dtype=torch.long)

    def class_weights(self):
        """Trả về class weights ngược nghịch để xử lý mất cân bằng."""
        counts = np.zeros(len(self.CLASS_NAMES))
        for _, lbl in self.samples:
            counts[lbl] += 1
        weights = counts.sum() / (len(counts) * counts)
        return torch.tensor(weights, dtype=torch.float32)

In [ ]:
# ==============================================================================
# 5. TRAINING LOGIC (CÓ TRACKING HISTORY, EARLY STOPPING & CONFUSION MATRIX)
# ==============================================================================
import seaborn as sns
from sklearn.metrics import confusion_matrix
from torch.utils.data import Subset

def train_one_epoch_cls(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = correct = total = 0
    for imgs, labels in tqdm(loader, desc="  train", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        if isinstance(logits, tuple): logits = logits[0] # Cho InceptionV3
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate_cls(model, loader, device, criterion=None):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    for imgs, labels in tqdm(loader, desc="  eval ", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        if isinstance(logits, tuple): logits = logits[0]

        if criterion is not None:
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)

        preds = logits.argmax(1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.tolist())

    avg_loss = total_loss / len(all_labels) if len(all_labels) > 0 else 0
    metrics = compute_cls_metrics(all_preds, all_labels)
    metrics["loss"] = avg_loss

    return metrics, all_preds, all_labels

def get_subset_class_weights(dataset, indices):
    """Tính class weights ĐỘC LẬP chỉ trên tập Train, chống Data Leakage"""
    counts = np.zeros(len(CLASS_NAMES))
    for idx in indices:
        _, lbl = dataset.samples[idx]
        counts[lbl] += 1
    weights = counts.sum() / (len(counts) * counts)
    return torch.tensor(weights, dtype=torch.float32)

def run_single_cls(model_key, epochs, batch_size, device, output_dir, val_ratio=0.15, patience=10):
    img_size = get_input_size(model_key)

    # Khởi tạo 2 dataset gốc với split khác biệt
    base_train_ds = BRISCClassificationDataset(CLS_TRAIN_DIR, split="train", image_size=img_size)
    base_val_ds   = BRISCClassificationDataset(CLS_TRAIN_DIR, split="test",  image_size=img_size)
    test_ds       = BRISCClassificationDataset(CLS_TEST_DIR,  split="test",  image_size=img_size)

    # Lấy toàn bộ nhãn (labels) từ tập train gốc để làm mốc Stratify
    all_labels = [label for _, label in base_train_ds.samples]
    indices = list(range(len(base_train_ds)))

    # Sử dụng train_test_split của sklearn với tham số stratify
    train_idx, val_idx = train_test_split(
        indices,
        test_size=val_ratio,
        random_state=42,
        stratify=all_labels
    )

    # Gắn index vào đúng gốc Dataset
    train_ds = Subset(base_train_ds, train_idx)
    val_ds   = Subset(base_val_ds, val_idx)

    print(f"    Train samples: {len(train_ds)} | Val samples: {len(val_ds)} | Test samples: {len(test_ds)}")

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS)
    val_dl   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)

    model = build_cls_model(model_key, num_classes=4, pretrained=True).to(device)

    # KHẮC PHỤC DATA LEAKAGE: Chỉ tính weight trên index của tập train
    cls_weights = get_subset_class_weights(base_train_ds, train_idx).to(device)
    criterion = nn.CrossEntropyLoss(weight=cls_weights)

    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    save_dir = os.path.join(output_dir, model_key)
    os.makedirs(save_dir, exist_ok=True)

    history = {"epoch": [], "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": [], "lr": []}

    # KHẮC PHỤC METRIC VÀ EARLY STOPPING
    best_f1, best_state = 0.0, deepcopy(model.state_dict())
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        current_lr = optimizer.param_groups[0]["lr"]

        tr_loss, tr_acc = train_one_epoch_cls(model, train_dl, criterion, optimizer, device)

        val_metrics, _, _ = evaluate_cls(model, val_dl, device, criterion)
        val_acc = val_metrics["weighted"]["accuracy"]
        val_f1  = val_metrics["weighted"]["f1"]

        scheduler.step()

        print(f"  epoch {epoch:>3}/{epochs} | Tr_Loss {tr_loss:.4f} | Tr_Acc {tr_acc:.4f} || Val_Loss {val_metrics['loss']:.4f} | Val_F1 {val_f1:.4f}")

        history["epoch"].append(epoch)
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(val_metrics['loss'])
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)
        history["lr"].append(current_lr)

        # So sánh dựa trên Weighted F1-Score
        if val_f1 > best_f1:
            best_f1 = val_f1
            epochs_no_improve = 0
            best_state = deepcopy(model.state_dict())
            torch.save(best_state, os.path.join(save_dir, "best_model.pt"))
        else:
            epochs_no_improve += 1

        # Kích hoạt Early Stopping
        if epochs_no_improve >= patience:
            print(f"  ⚠ Early stopping kích hoạt tại epoch {epoch} do Val_F1 không tăng trong {patience} vòng liên tiếp.")
            break

    model.load_state_dict(best_state)
    test_result, test_preds, test_labels = evaluate_cls(model, test_dl, device, criterion)

    # Lưu History
    pd.DataFrame(history).to_csv(os.path.join(save_dir, "training_history.csv"), index=False)

    # ---------------------------------------------------------
    # VẼ LEARNING CURVES (3 BIỂU ĐỒ)
    # ---------------------------------------------------------
    plt.figure(figsize=(18, 5))

    # Biểu đồ 1: Loss
    plt.subplot(1, 3, 1)
    plt.plot(history["epoch"], history["train_loss"], label="Train Loss", color='blue')
    plt.plot(history["epoch"], history["val_loss"], label="Val Loss", color='orange')
    plt.title(f"{model_key} - Loss Curve"); plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.legend(); plt.grid(True, linestyle='--', alpha=0.6)

    # Biểu đồ 2: Accuracy
    plt.subplot(1, 3, 2)
    plt.plot(history["epoch"], history["train_acc"], label="Train Acc", color='blue')
    plt.plot(history["epoch"], history["val_acc"], label="Val Acc", color='orange')
    plt.title(f"{model_key} - Accuracy"); plt.xlabel("Epoch"); plt.ylabel("Accuracy")
    plt.legend(); plt.grid(True, linestyle='--', alpha=0.6)

    # Biểu đồ 3: F1-Score
    plt.subplot(1, 3, 3)
    plt.plot(history["epoch"], history["val_f1"], label="Val F1 (Weighted)", color='green')
    plt.title(f"{model_key} - Validation F1-Score"); plt.xlabel("Epoch"); plt.ylabel("F1-Score")
    plt.legend(); plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "learning_curves.png"))
    plt.close()

    # CONFUSION MATRIX
    cm = confusion_matrix(test_labels, test_preds)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel('Predicted Class'); plt.ylabel('Actual Class'); plt.title(f'{model_key} - Confusion Matrix')
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "confusion_matrix.png")); plt.close()

    with open(os.path.join(save_dir, "final_metrics.json"), "w") as f:
        json.dump(test_result, f, indent=2)

    return test_result

In [ ]:
'''
# ==============================================================================
# 6. BẢNG ĐIỀU KHIỂN CHẠY THÍ NGHIỆM
# ==============================================================================
if __name__ == "__main__":
    # -------- ĐIỀU CHỈNH CÁC THÔNG SỐ NÀY --------
    MODEL_TO_RUN   = "resnet50"   # "resnet50", "resnet101", "densenet121", "densenet169", "mobilenetv2_100", "mobilenetv3_large_100",
                                  # "efficientnet_b0", "efficientnet_b1", "efficientnet_b2", "xception", "vgg16", "vgg19", "inception_v3"
    EPOCHS         = 50           # Đặt = 2 để chạy test, = 50 để train thật
    NUM_RUNS       = 3            # 3 lần lặp để tính Mean ± Std
    # ---------------------------------------------

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Thiết bị: {device.type.upper()}")

    models_to_run = [MODEL_TO_RUN] if MODEL_TO_RUN else list(TIMM_NAME_MAP.keys())
    all_results = {}

    for model_key in models_to_run:
        print(f"\n{'#'*80}")
        print(f" BẮT ĐẦU CHIẾN DỊCH: {model_key.upper()} ({NUM_RUNS} RUNS × {EPOCHS} EPOCHS)")
        print(f"{'#'*80}")

        run_results = []

        for run_idx in range(1, NUM_RUNS + 1):
            seed = 42 + (run_idx * 7) # Đổi seed: 49, 56, 63...
            seed_everything(seed)
            print(f"\n{'─'*50}\n  ➤ ĐANG CHẠY RUN {run_idx}/{NUM_RUNS} (Seed: {seed})\n{'─'*50}")

            try:
                # Gọi hàm chạy đơn lẻ (Hệ thống sẽ chạy đủ 30 epochs)
                result = run_single_cls(
                    model_key,
                    epochs=EPOCHS,
                    batch_size=CLS_BATCH_SIZE,
                    device=device,
                    output_dir=OUTPUT_DIR
                )

                # Sửa tên file biểu đồ để không bị ghi đè giữa các Run
                save_dir = os.path.join(OUTPUT_DIR, model_key)
                if os.path.exists(os.path.join(save_dir, "learning_curves.png")):
                    os.rename(os.path.join(save_dir, "learning_curves.png"), os.path.join(save_dir, f"learning_curves_run{run_idx}.png"))
                if os.path.exists(os.path.join(save_dir, "best_model.pt")):
                    os.rename(os.path.join(save_dir, "best_model.pt"), os.path.join(save_dir, f"best_model_run{run_idx}.pt"))

                run_results.append(result)
            except Exception as e:
                print(f"  ✗ Gặp lỗi ở Run {run_idx}: {e}")

        # TỔNG HỢP VÀ IN BẢNG NẾU CHẠY ĐỦ RUNS
        if run_results:
            agg_results = aggregate_runs(run_results)
            all_results[model_key] = agg_results

            # In ra màn hình bảng y hệt bài báo
            print_cls_table(model_key, agg_results)

            # Lưu file JSON chứa thông số Mean ± Std
            with open(os.path.join(OUTPUT_DIR, f"{model_key}_Aggregated.json"), "w") as f:
                json.dump(agg_results, f, indent=2)

    # Lưu tổng hợp toàn bộ các models đã chạy
    with open(os.path.join(OUTPUT_DIR, "Final_Table4_Results.json"), "w") as f:
        json.dump(all_results, f, indent=2)

    print("\n✅ CHIẾN DỊCH HOÀN TẤT! Đã đồng bộ lên Google Drive.")
'''

'\n# ==============================================================================\n# 6. BẢNG ĐIỀU KHIỂN CHẠY THÍ NGHIỆM\n# ==============================================================================\nif __name__ == "__main__":\n    # -------- ĐIỀU CHỈNH CÁC THÔNG SỐ NÀY --------\n    MODEL_TO_RUN   = "resnet50"   # "resnet50", "resnet101", "densenet121", "densenet169", "mobilenetv2_100", "mobilenetv3_large_100",\n                                  # "efficientnet_b0", "efficientnet_b1", "efficientnet_b2", "xception", "vgg16", "vgg19", "inception_v3"\n    EPOCHS         = 50           # Đặt = 2 để chạy test, = 50 để train thật\n    NUM_RUNS       = 3            # 3 lần lặp để tính Mean ± Std\n    # ---------------------------------------------\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    print(f"🚀 Thiết bị: {device.type.upper()}")\n\n    models_to_run = [MODEL_TO_RUN] if MODEL_TO_RUN else list(TIMM_NAME_MAP.keys())\n    all_results = {}\n\n    fo

In [ ]:
# ==============================================================================
# 6. BẢNG ĐIỀU KHIỂN CHẠY THÍ NGHIỆM (BẢN 1 RUN TIẾT KIỆM TÀI NGUYÊN)
# ==============================================================================
if __name__ == "__main__":
    # -------- ĐIỀU CHỈNH CÁC THÔNG SỐ NÀY --------
    MODEL_TO_RUN   = "inception_v3"   # "resnet50", "resnet101", "densenet121", "densenet169", "mobilenetv2_100", "mobilenetv3_large_100",
                                  # "efficientnet_b0", "efficientnet_b1", "efficientnet_b2", "xception", "vgg16", "vgg19", "inception_v3"
    EPOCHS         = 50           # Đặt = 2 để chạy test, = 50 để train thật
    # ---------------------------------------------

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Thiết bị: {device.type.upper()}")

    models_to_run = [MODEL_TO_RUN] if MODEL_TO_RUN else list(TIMM_NAME_MAP.keys())
    all_results = {}

    for model_key in models_to_run:
        print(f"\n{'#'*80}")
        print(f" BẮT ĐẦU CHIẾN DỊCH: {model_key.upper()} (1 RUN × {EPOCHS} EPOCHS)")
        print(f"{'#'*80}")

        try:
            # Gọi hàm chạy trực tiếp
            result = run_single_cls(
                model_key,
                epochs=EPOCHS,
                batch_size=CLS_BATCH_SIZE,
                device=device,
                output_dir=OUTPUT_DIR
            )

            all_results[model_key] = result

            # In ra màn hình bảng kết quả
            print_cls_table(model_key, result)

        except Exception as e:
            print(f"  ✗ Gặp lỗi ở mô hình {model_key}: {e}")

    # Lưu tổng hợp toàn bộ các models đã chạy
    with open(os.path.join(OUTPUT_DIR, "Final_Table4_Results.json"), "w") as f:
        json.dump(all_results, f, indent=2)

    print("\n✅ CHIẾN DỊCH HOÀN TẤT! Đã đồng bộ lên Google Drive.")

🚀 Thiết bị: CUDA

################################################################################
 BẮT ĐẦU CHIẾN DỊCH: INCEPTION_V3 (1 RUN × 50 EPOCHS)
################################################################################
    Train samples: 4250 | Val samples: 750 | Test samples: 1000


model.safetensors:   0%|          | 0.00/95.5M [00:00<?, ?B/s]

  epoch   1/50 | Tr_Loss 0.4733 | Tr_Acc 0.8240 || Val_Loss 0.1771 | Val_F1 0.9315


  epoch   2/50 | Tr_Loss 0.2303 | Tr_Acc 0.9106 || Val_Loss 0.1174 | Val_F1 0.9599


  epoch   3/50 | Tr_Loss 0.1347 | Tr_Acc 0.9501 || Val_Loss 0.0856 | Val_F1 0.9718


...

  epoch  23/50 | Tr_Loss 0.0219 | Tr_Acc 0.9927 || Val_Loss 0.0932 | Val_F1 0.9839


  epoch  24/50 | Tr_Loss 0.0207 | Tr_Acc 0.9911 || Val_Loss 0.0737 | Val_F1 0.9853


  epoch  25/50 | Tr_Loss 0.0181 | Tr_Acc 0.9936 || Val_Loss 0.0750 | Val_F1 0.9853


  epoch  26/50 | Tr_Loss 0.0166 | Tr_Acc 0.9946 || Val_Loss 0.0674 | Val_F1 0.9826
  ⚠ Early stopping kích hoạt tại epoch 26 do Val_F1 không tăng trong 10 vòng liên tiếp.



  KẾT QUẢ TEST: INCEPTION_V3
  Class                Precision         Recall       F1-Score       Accuracy
  -----------------------------------------------------------------------
  glioma                  99.21%         98.43%         98.81%              —
  meningioma              99.01%         98.37%         98.69%              —
  no_tumor               100.00%        100.00%        100.00%              —
  pituitary               98.68%        100.00%         99.34%              —
  macro                   99.23%         99.20%         99.21%              —
  weighted                99.10%         99.10%         99.10%         99.10%


✅ CHIẾN DỊCH HOÀN TẤT! Đã đồng bộ lên Google Drive.
